In [ ]:
# CELULA 1 -- Imports, Variáveis de Ambiente, GCP Client e Configurações Globais

import os
import io
import re
import requests
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import storage

# 1. Autenticação Local GCP
BASE_DIR = os.path.dirname(os.path.abspath(""))
KEY_PATH = os.path.join(BASE_DIR, "config", "gcp_key.json")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

# 2. Instanciação do Client do Google Cloud Storage
client = storage.Client.from_service_account_json(KEY_PATH)

# 3. Parâmetros Gerais
ANO_ALVO = "2026"
BUCKET_BRONZE = "cvm-landing-zone-bronze-dev"
BUCKET_SILVER = "cvm-lakehouse-silver-dev"

pd.set_option("display.max_columns", None)

print("[+] Client GCP inicializado com sucesso!")
print(f"[+] Target Bronze: gs://{BUCKET_BRONZE}")
print(f"[+] Target Silver: gs://{BUCKET_SILVER}")

In [ ]:
# CELULA 2 -- Funções Utilitárias de Storage e Diagnóstico de Lacunas

bucket = client.bucket(BUCKET_BRONZE)

def upload_in_memory(url: str, gcs_path: str, content_type: str) -> bool:
    """Faz o streaming do download direto para o GCS em memória."""
    try:
        response = requests.get(url, timeout=90)
        if response.status_code == 200:
            blob = bucket.blob(gcs_path)
            blob.upload_from_string(response.content, content_type=content_type)
            print(f"[+] Success: gs://{BUCKET_BRONZE}/{gcs_path}")
            return True
        elif response.status_code == 404:
            print(f"[-] Arquivo não encontrado na origem CVM (404): {url}")
            return False
        else:
            print(f"[!] Erro HTTP {response.status_code} para {url}")
            return False
    except Exception as e:
        print(f"[!] Exceção ao processar {url}: {str(e)}")
        return False

def gerar_tabela_calendario_dias_uteis(ano: str) -> list:
    """Gera lista de dias úteis em formato YYYYMMDD do início do ano até ontem."""
    hoje = datetime.now().date()
    ontem = hoje - timedelta(days=1)
    dias_uteis = pd.bdate_range(start=f"{ano}-01-01", end=ontem)
    return dias_uteis.strftime("%Y%m%d").tolist()

def obter_datas_faltantes_gcs(prefixo_gcs: str, sufixo_extensao: str = ".csv", ano_alvo: str = "2026") -> list:
    """Calcula a diferença entre os dias úteis esperados e os arquivos salvos no GCS."""
    datas_esperadas = set(gerar_tabela_calendario_dias_uteis(ano_alvo))
    blobs = client.list_blobs(BUCKET_BRONZE, prefix=prefixo_gcs)
    
    datas_existentes = set()
    padrao_data = re.compile(r"(\d{8})")
    
    for blob in blobs:
        if blob.name.endswith(sufixo_extensao):
            match = padrao_data.search(blob.name)
            if match:
                datas_existentes.add(match.group(1))
                
    datas_faltantes = sorted(list(datas_esperadas - datas_existentes))
    
    print(f"\n[==>] DIAGNÓSTICO DE INGESTÃO [{prefixo_gcs}]")
    print(f"[-] Dias úteis esperados em {ano_alvo}: {len(datas_esperadas)}")
    print(f"[+] Fotos cadastradas na Bronze: {len(datas_existentes)}")
    print(f"[!] Fotos pendentes de ingestão: {len(datas_faltantes)}")
    
    return datas_faltantes

In [ ]:
# CELULA 3 -- Execução Incremental Idempotente do Cadastro (CAD_FI) com Orquestrador main()

def executar_ingestao_cad_fi() -> bool:
    """
    Orquestra a verificação de lacunas e realiza o download incremental do cadastro de fundos (CAD_FI)
    da CVM, mantendo a foto atualizada na raiz do bucket Bronze e o histórico nas partições diárias.
    """
    print("================================================================================")
    print("[==>] INICIANDO INGESTÃO INCREMENTAL DO CADASTRO (CAD_FI)")
    print("================================================================================\n")

    try:
        # 1. Identifica os dias úteis que faltam no bucket Bronze
        datas_cad_pendentes = obter_datas_faltantes_gcs(
            prefixo_gcs=f"cvm/cad_fi/ano={ANO_ALVO}/", 
            sufixo_extensao=".csv",
            ano_alvo=ANO_ALVO
        )

        # 2. Executa a ingestão se houver lacunas
        if not datas_cad_pendentes:
            print("\n[+] Landing Zone 100% atualizada para todos os dias úteis do ano!")
            return True

        print(f"\n[+] Baixando foto cadastral para {len(datas_cad_pendentes)} data(s) pendente(s)...")
        url_cad = "http://dados.cvm.gov.br/dados/FI/CAD/DADOS/cad_fi.csv"
        path_cad_raiz = "cvm/cad_fi/cad_fi.csv"
        
        # Faz o download da versão mais recente e atualiza o espelho na raiz para consumo do Silver
        sucesso_raiz = upload_in_memory(url_cad, path_cad_raiz, "text/csv")
        
        if sucesso_raiz:
            # Preenche as partições históricas de dias úteis pendentes
            for data_str in datas_cad_pendentes:
                mes_str = data_str[4:6]
                path_cad_particao = f"cvm/cad_fi/ano={ANO_ALVO}/mes={mes_str}/cad_fi_{data_str}.csv"
                upload_in_memory(url_cad, path_cad_particao, "text/csv")
                
            print("\n[+] Cadastro CVM atualizado na raiz e particionado por data com sucesso!")
            return True
        else:
            print("\n[!] Falha ao obter a foto cadastral na origem CVM.")
            return False

    except Exception as e:
        print(f"\n[!] Erro crítico durante a execução da ingestão do CAD_FI: {e}")
        return False


def main():
    """
    Função principal orquestradora do script de ingestão do CAD_FI na Camada Bronze.
    """
    executar_ingestao_cad_fi()


if __name__ == "__main__":
    main()